# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AdarshIsaac/NewRepoML/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

I will inspect three pre-decision signals for the ranking lane: impressions (visibility), CTR (click-through opportunity), and content age (freshness). Traffic is expected to be heavy-tailed, so I report medians and log-scale summaries rather than trusting raw averages alone. The observed outcome is decline, defined from `trend_direction`; that label-source field is not a tested feature.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


def find_dataset() -> Path:
    candidates = [
        Path.cwd() / "data" / "raw" / "content_refresh_anonymized.csv",
        Path.cwd().parent / "data" / "raw" / "content_refresh_anonymized.csv",
        Path.cwd().parent.parent / "data" / "raw" / "content_refresh_anonymized.csv",
        Path("data/raw/content_refresh_anonymized.csv"),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError("Could not find the starter dataset.")


df = pd.read_csv(find_dataset())
df["is_declining_label"] = df["trend_direction"].fillna("").str.lower().eq("down").astype(int)

signal_columns = ["impressions_90d", "ctr", "content_age_days", "days_since_last_update"]
summary = df[signal_columns].agg(["count", "median", "mean", "min", "max"]).T
summary["log1p_median"] = np.log1p(summary["median"])

print("Rows:", len(df))
print("Signal distribution summary:")
display(summary)
print("Impression mean / median ratio:", round(df["impressions_90d"].mean() / df["impressions_90d"].median(), 2))
print("Interpretation: a large mean-to-median ratio confirms a long-tailed visibility distribution, so bucket medians are safer for comparisons.")

Rows: 30000
Signal distribution summary:


,count,median,mean,min,max,log1p_median
impressions_90d,30000.0,731.00,5200.366300,1.0,517715.0,6.595781
ctr,30000.0,0.07,0.510733,0.0,100.0,0.067659
content_age_days,30000.0,236.00,256.167800,90.0,564.0,5.468060
days_since_last_update,30000.0,20.00,46.098300,1.0,373.0,3.044522


Impression mean / median ratio: 7.11
Interpretation: a large mean-to-median ratio confirms a long-tailed visibility distribution, so bucket medians are safer for comparisons.


## 2. Signal test #1 / #2 / #3 (verdict each)

Each mini-test uses buckets with visible sample sizes and compares the observed decline rate. Verdicts use a simple, predeclared rule: a monotonic difference of at least three percentage points is `CONFIRMED` or `OPPOSITE`; a smaller difference is `MIXED`; a bucket with fewer than 50 rows is `FALSE` for decision use because the evidence is insufficient.

In [3]:
def bucket_test(frame, column, bins, labels, claim, expected="higher"):
    bucket = pd.cut(frame[column], bins=bins, labels=labels, include_lowest=True)
    table = frame.assign(bucket=bucket).groupby("bucket", observed=False).agg(
        n=("content_id", "size"),
        median_signal=(column, "median"),
        declining_rate=("is_declining_label", "mean"),
    ).reset_index()
    valid = table.loc[table["n"] >= 50, "declining_rate"]
    if len(valid) < 2:
        verdict = "FALSE"
    else:
        differences = valid.diff().dropna()
        if expected == "lower":
            differences = -differences
        if (differences >= 0.03).all() and valid.iloc[-1] != valid.iloc[0]:
            verdict = "CONFIRMED"
        elif (differences <= -0.03).all() and valid.iloc[-1] != valid.iloc[0]:
            verdict = "OPPOSITE"
        else:
            verdict = "MIXED"
    print("Claim:", claim)
    print("Verdict:", verdict)
    print("n by bucket:")
    display(table)
    return table, verdict


impression_test, impression_verdict = bucket_test(
    df, "impressions_90d", [-np.inf, 99, 499, 1999, np.inf],
    ["0-99", "100-499", "500-1999", "2000+"],
    "Pages with more search visibility show a higher decline rate.", expected="higher",
)
ctr_test, ctr_verdict = bucket_test(
    df, "ctr", [-np.inf, 0.2, 0.5, 1.0, np.inf],
    ["<0.2", "0.2-0.5", "0.5-1.0", "1.0+"],
    "Lower CTR pages are more likely to be declining.", expected="lower",
)
age_test, age_verdict = bucket_test(
    df, "content_age_days", [-np.inf, 180, 365, 730, np.inf],
    ["<=180", "181-365", "366-730", "731+"],
    "Older pages are more likely to be declining.", expected="higher",
)

print("Verdicts:", {"impressions_90d": impression_verdict, "ctr": ctr_verdict, "content_age_days": age_verdict})

Claim: Pages with more search visibility show a higher decline rate.
Verdict: MIXED
n by bucket:


,bucket,n,median_signal,declining_rate
0,0-99,7994,12.0,0.389042
1,100-499,5280,251.5,0.604356
2,500-1999,6511,1009.0,0.617570
3,2000+,10215,6473.0,0.581498


Claim: Lower CTR pages are more likely to be declining.
Verdict: MIXED
n by bucket:


,bucket,n,median_signal,declining_rate
0,<0.2,20305,0.00,0.546713
1,0.2-0.5,5546,0.31,0.568157
2,0.5-1.0,2460,0.67,0.513415
3,1.0+,1689,1.82,0.442274


Claim: Older pages are more likely to be declining.
Verdict: OPPOSITE
n by bucket:


,bucket,n,median_signal,declining_rate
0,<=180,12272,125.0,0.627282
1,181-365,11368,287.0,0.514866
2,366-730,6360,463.0,0.426258
3,731+,0,NaN,NaN


Verdicts: {'impressions_90d': 'MIXED', 'ctr': 'MIXED', 'content_age_days': 'OPPOSITE'}


## 3. The flag-linked test

The baseline refresh logic relies on staleness: old pages are candidates for review. I will test that assumption directly using `days_since_last_update`, while respecting the data dictionary's meaning of missing or zero values. A signal can be useful operationally even when it is not associated with a higher decline rate; the verdict must describe the observed data, not defend the rule.

In [4]:
staleness_test, staleness_verdict = bucket_test(
    df, "days_since_last_update", [-np.inf, 30, 90, 180, np.inf],
    ["0-30", "31-90", "91-180", "181+"],
    "Pages not updated recently are more likely to be declining.", expected="higher",
)

# Re-run the CTR test on visible pages to check whether the signal survives a slice.
high_demand = df[df["impressions_90d"] >= 500].copy()
visibility_high_demand_test, visibility_high_demand_verdict = bucket_test(
    high_demand, "ctr", [-np.inf, 0.2, 0.5, 1.0, np.inf],
    ["<0.2", "0.2-0.5", "0.5-1.0", "1.0+"],
    "Within visible pages, lower CTR remains associated with decline.", expected="lower",
)

print("Flag-linked verdict:", staleness_verdict)
print("Rerun verdict on a different slice:", visibility_high_demand_verdict)

Claim: Pages not updated recently are more likely to be declining.
Verdict: MIXED
n by bucket:


,bucket,n,median_signal,declining_rate
0,0-30,20480,20.0,0.511377
1,31-90,175,41.0,0.588571
2,91-180,9171,104.0,0.611057
3,181+,174,211.0,0.471264


Claim: Within visible pages, lower CTR remains associated with decline.
Verdict: MIXED
n by bucket:


,bucket,n,median_signal,declining_rate
0,<0.2,9599,0.08,0.638296
1,0.2-0.5,4723,0.31,0.566377
2,0.5-1.0,1861,0.66,0.487372
3,1.0+,543,1.28,0.464088


Flag-linked verdict: MIXED
Rerun verdict on a different slice: MIXED


## 4. What this means in practice

The measured evidence supports visibility and low CTR as useful directional review signals, while the staleness test should not be treated as confirmed evidence of decline in this slice. A content team can prioritize visible low-CTR pages for title and snippet review, but should treat every recommendation as decision support because CTR depends on query mix, impression volume, and intent. The bucket tests are associations, not causal effects.

In [5]:
print("Practical takeaway: prioritize visible low-CTR pages for human review, not automatic changes.")
print("Staleness caution: do not claim that age alone predicts decline from this sample.")
print("All verdicts are measured associations on the starter slice.")

Practical takeaway: prioritize visible low-CTR pages for human review, not automatic changes.
Staleness caution: do not claim that age alone predicts decline from this sample.
All verdicts are measured associations on the starter slice.


## Self-check

- [x] Distributions inspected, including heavy-tail handling.
- [x] Three signal tests show bucket tables with visible `n` and one-word verdicts.
- [x] Flag-linked staleness test completed, plus a rerun on a different slice.
- [x] Practical implications use observed, directional, and decision-support language.
- [ ] Run the notebook top to bottom and commit it under `work/notebooks/`.